# CTB ProSiT — minimal reproduction

This notebook loads the saved Petri net and ProSiT parameters, verifies every file, runs the published what-if models, and reconstructs the numerical thesis claims from the shipped per-seed evidence.

The confidential terminal event log is not included. Therefore the historical hold-out comparison is **arithmetically reproduced from frozen per-seed validation outputs**, while the saved-model what-if experiment is rerun exactly. This boundary is deliberate and auditable.


## 1. Install the frozen environment

Run All is sufficient. Python 3.11 or 3.12 is recommended.


In [ ]:
%pip install -r requirements.txt --disable-pip-version-check -q


## 2. Verify and load the saved models


In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, ".")
import reviewer_runner as rr

integrity = rr.verify_package_files()
print(f"Integrity PASS: {len(integrity)} frozen files verified")
models = rr.load_analysis_models()
display(rr.model_summary(models))


## Why PNML, JSON, and pickle are all provided

- **PNML** contains the portable Petri-net control flow.
- **JSON** makes the parameters human-readable and follows ProSiT's documented `to_json()` / `from_json()` interface.
- **Verified pickle** is authoritative for exact execution. In ProSiT 1.0.3, JSON does not preserve the empirical samples stored in decision-rule leaves and the CTB tuple keys containing `nan` cannot be read back reliably. JSON alone would lose runtime state.


In [ ]:
scenario_models = rr.load_models()
json_report = rr.export_and_reload_official_json(
    scenario_models["baseline"], Path("outputs/json_api_demo.json")
)
display(pd.Series(json_report))


## 3. Verify the original and what-if model contracts


In [ ]:
display(rr.assert_model_contracts(scenario_models))
display(rr.parameter_component_inventory(scenario_models["baseline"]).head(20))


## 4. Reconstruct the historical three-state ablation

This recomputes means, 95% t-intervals, and paired seed contrasts from the ten shipped replication rows for `no_rules`, `rules_only`, and `rules_workload`.


In [ ]:
historical_summary, historical_contrasts = rr.reconstruct_historical_ablation()
display(historical_summary)
display(historical_contrasts)


## 5. Inspect the remaining claim evidence


In [ ]:
evidence = rr.load_claim_evidence()
print("Available evidence:", ", ".join(evidence))
display(evidence["bottleneck_ranking"].head(8))
display(evidence["capacity_by_block"].head(8))
display(evidence["scenario_state_ablation"].query(
    "metric in ['mean_turnaround_min', 'mean_rmg_service_min', 'mean_rmg_pre_service_min']"
))


## 6. Exact saved-model scenario reproduction

The default reruns 10 matched seeds × 3 saved models × 17,892 cases and compares every regenerated table with the frozen thesis output. It took about 30 minutes on the author's computer. Set `RUN_FULL_SCENARIOS = False` only for a short mechanics check; a smoke run is not a numerical reproduction.


In [ ]:
RUN_FULL_SCENARIOS = True

mode = "full" if RUN_FULL_SCENARIOS else "smoke"
output_dir = rr.run_saved_models(mode=mode)
print(f"Fresh outputs: {output_dir}")

if RUN_FULL_SCENARIOS:
    comparison = rr.compare_with_frozen_results(output_dir)
    display(comparison)
    assert comparison["values_match"].all()
    print("FULL REPRODUCTION PASS — all scenario tables match.")
else:
    print("SMOKE PASS — execution works; frozen thesis numbers were not rerun.")


## Interpretation boundary

The saved model is reproducible and its interventions are executable. This does not make the scenario effects causal estimates for the physical terminal. The current model lacks explicit container locations, crane trajectories, physical transit, and queue states; the notebook exposes those limits alongside the successful checks.
